# PPG raw -> HRV batch analysis

Notebook này xử lý **PPG raw** đã xuất từ Layer 2 (`ppg_raw.edf`, 100Hz,
block dài nhất mỗi session) và tính các chỉ số HRV chuẩn, theo **cả 2 mức**:

- **OVERALL**: toàn bộ block PPG của session.
- **Theo từng event** (`REST_BEFORE` / `MEDITATION` / `REST_AFTER`...) --
  đọc mốc thời gian từ `event_labels_block.csv` (do `lib/event_labels.py`
  tạo ở Layer 2, latency tính theo sample EEG 244Hz) rồi quy đổi sang
  sample PPG 100Hz để cắt đúng đoạn.

## Pipeline xử lý (xem `lib/ppg_hrv.py`)

1. Bandpass 0.5-8Hz (giống `filter_ppg()` trong module QC Layer 2).
2. Xác định đỉnh tâm thu (`scipy.signal.find_peaks`, prominence + khoảng cách tối thiểu 0.35s).
3. IBI (inter-beat interval) giữa các đỉnh liên tiếp, lọc theo khoảng sinh lý hợp lệ 0.3-1.8s (33-200 bpm).
4. **Miền thời gian**: HR trung bình, RMSSD, SDRR (=SDNN), pRR50, pRR20.
5. **Miền tần số**: nội suy IBI thành tachogram đều mẫu 4Hz, Welch PSD, công
   suất LF (0.04-0.15Hz) / HF (0.15-0.4Hz) -> tỉ lệ LF/HF. Chỉ tính khi đoạn
   có **>= 60 giây** IBI liên tục, ngắn hơn thì để `NaN` thay vì 1 con số
   không đáng tin (rest trước/sau thường quá ngắn cho LF/HF, thiền thì đủ dài).

## Output

Bảng dài (long format), 1 dòng / session / phase, có đủ khoá
`participant_key`, `study_day`, `session_id` để **join với bảng survey**
(`session_analysis_primary.csv`) ở bước phân tích sau -- không tự join ở
đây, chỉ xuất metadata sạch, đúng khoá.


In [ ]:
from pathlib import Path
import re
import sys
import traceback

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Notebook nằm ở notebooks/ nên gốc project cách 2 cấp thư mục.
REPO_ROOT = Path("..").resolve()

# import kiểu flat module (không qua package lib.) -- khớp cách Layer 2 tự
# thêm lib/ vào sys.path, tránh phụ thuộc lib/__init__.py.
LIB_DIR = REPO_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import ppg_hrv
import importlib
importlib.reload(ppg_hrv)

print("ppg_hrv module:", ppg_hrv.__file__)

## 1. Cấu hình

In [ ]:
LAYER2_DIR = REPO_ROOT / "data" / "analytics" / "layer2"
LAYER1_SESSIONS_PATH = REPO_ROOT / "data" / "analytics" / "layer1" / "sessions.csv"

OUT_DIR = REPO_ROOT / "data" / "analytics" / "ppg_hrv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EEG_FS = 244.0       # sample rate latency trong event_labels_block.csv (Layer 2)
PPG_FS = 100.0       # sample rate ppg_raw.edf (OPTICAL_FS)

PARTICIPANT_RE = re.compile(r"participant=([^/\\]+)")
STUDY_DAY_RE = re.compile(r"study_day=([^/\\]+)")
SESSION_RE = re.compile(r"session_id=([^/\\]+)")

print("LAYER2_DIR:", LAYER2_DIR, "exists:", LAYER2_DIR.exists())
print("OUT_DIR:", OUT_DIR)

## 2. Quét toàn bộ session có `ppg_raw.edf`

In [ ]:
def parse_identity(ppg_path: Path) -> dict:
    parts = str(ppg_path)
    participant = PARTICIPANT_RE.search(parts)
    study_day = STUDY_DAY_RE.search(parts)
    session_id = SESSION_RE.search(parts)
    return {
        "participant_key": participant.group(1) if participant else "unknown",
        "study_day": int(study_day.group(1)) if study_day and study_day.group(1).isdigit() else None,
        "session_id": session_id.group(1) if session_id else ppg_path.stem,
    }

ppg_files = sorted(LAYER2_DIR.rglob("ppg_raw.edf"))
print(f"Tìm thấy {len(ppg_files)} file ppg_raw.edf")

sessions_l1 = pd.read_csv(LAYER1_SESSIONS_PATH, low_memory=False) if LAYER1_SESSIONS_PATH.exists() else pd.DataFrame()
role_lookup = (
    sessions_l1.set_index("session_id")["session_role"].to_dict()
    if not sessions_l1.empty and "session_role" in sessions_l1.columns
    else {}
)

session_index = []
for ppg_path in ppg_files:
    identity = parse_identity(ppg_path)
    event_csv = ppg_path.with_name("event_labels_block.csv")
    session_index.append({
        **identity,
        "ppg_path": ppg_path,
        "event_csv_path": event_csv if event_csv.exists() else None,
        "session_role": role_lookup.get(identity["session_id"]),
    })

session_index_df = pd.DataFrame(session_index)
print(f"Có event_labels_block.csv: {session_index_df['event_csv_path'].notna().sum()}/{len(session_index_df)}")
session_index_df[["participant_key", "study_day", "session_id", "session_role"]].head(10)

## 3. Hàm phân tích 1 session (OVERALL + theo từng event)

`event_labels_block.csv` ghi `latency`/`duration_samples` theo sample **EEG**
(244Hz, xem `lib/event_labels.py`). PPG xuất riêng ở **100Hz** (`OPTICAL_FS`)
nhưng cùng mốc thời gian thực (cùng block gốc) -- quy đổi qua giây rồi nhân
lại với PPG_FS để cắt đúng đoạn PPG tương ứng.

Riêng phase `MEDITATION` được chia thêm thành các cửa sổ liên tiếp ~2 phút (`MEDITATION_WIN01`, `MEDITATION_WIN02`, ...; `lib/ppg_hrv.py::split_windows()`) để xem HRV thay đổi ra sao TRONG LÚC thiền, không chỉ 1 số trung bình cho cả đoạn. Cửa sổ cuối có thể ngắn hơn 2 phút (phần dư phép chia) -- vẫn giữ lại, `hrv_quality_flag` sẽ tự đánh dấu SHORT_SEGMENT nếu quá ngắn.

In [ ]:
MEDITATION_WINDOW_S = 120.0  # chia MEDITATION thành các cửa sổ ~2 phút để xem HRV biến đổi trong lúc thiền, không chỉ 1 số trung bình


def analyze_session(ppg_path: Path, event_csv_path, identity: dict) -> list[dict]:
    rows = []

    raw = mne.io.read_raw_edf(ppg_path, preload=True, verbose=False)
    ppg = raw.get_data()[0]
    fs = float(raw.info["sfreq"])

    base_meta = {
        "participant_key": identity["participant_key"],
        "study_day": identity["study_day"],
        "session_id": identity["session_id"],
        "session_role": identity["session_role"],
    }

    # --- OVERALL: toàn bộ block PPG ---
    overall = ppg_hrv.analyze_segment(ppg, fs)
    rows.append({**base_meta, "phase": "OVERALL", **overall})

    # --- Theo từng event (nếu có) ---
    if event_csv_path is not None:
        events = pd.read_csv(event_csv_path)
        for _, ev in events.iterrows():
            start_s = ev["latency"] / EEG_FS
            end_s = (ev["latency"] + ev["duration_samples"]) / EEG_FS
            ppg_start = max(0, int(round(start_s * fs)))
            ppg_end = min(len(ppg), int(round(end_s * fs)))
            segment = ppg[ppg_start:ppg_end]

            metrics = ppg_hrv.analyze_segment(segment, fs)
            rows.append({
                **base_meta,
                "phase": ev["description"],
                "phase_start_s": float(start_s),
                "phase_end_s": float(end_s),
                **metrics,
            })

            # --- Chia riêng MEDITATION thành các cửa sổ ~2 phút liên tiếp ---
            # (KHÔNG áp dụng cho REST_BEFORE/REST_AFTER -- các phase đó đã
            # ngắn sẵn, chia nhỏ thêm sẽ dưới ngưỡng đủ dữ liệu cho LF/HF).
            if ev["description"] == "MEDITATION":
                windows = ppg_hrv.split_windows(start_s, end_s, window_s=MEDITATION_WINDOW_S)
                for win_idx, (w_start, w_end) in enumerate(windows, start=1):
                    w_ppg_start = max(0, int(round(w_start * fs)))
                    w_ppg_end = min(len(ppg), int(round(w_end * fs)))
                    win_segment = ppg[w_ppg_start:w_ppg_end]
                    win_metrics = ppg_hrv.analyze_segment(win_segment, fs)
                    rows.append({
                        **base_meta,
                        "phase": f"MEDITATION_WIN{win_idx:02d}",
                        "phase_start_s": float(w_start),
                        "phase_end_s": float(w_end),
                        "window_index": win_idx,
                        "window_start_min": float(w_start - start_s) / 60.0,
                        "window_end_min": float(w_end - start_s) / 60.0,
                        **win_metrics,
                    })

    return rows

## 4. Chạy batch

In [ ]:
from tqdm.notebook import tqdm

all_rows = []
run_log = []

for _, row in tqdm(session_index_df.iterrows(), total=len(session_index_df), desc="PPG HRV"):
    identity = row.to_dict()
    try:
        result_rows = analyze_session(row["ppg_path"], row["event_csv_path"], identity)
        all_rows.extend(result_rows)
        run_log.append({"session_id": row["session_id"], "status": "OK", "n_phases": len(result_rows)})
    except Exception as exc:
        run_log.append({
            "session_id": row["session_id"], "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
        })
        print(f"  [ERROR] {row['participant_key']} day={row['study_day']} {row['session_id']}: {exc}")

run_log_df = pd.DataFrame(run_log)
print("\nRun status:")
print(run_log_df["status"].value_counts().to_string())

## 5. Gắn cờ chất lượng + xuất kết quả

In [ ]:
ppg_hrv_long = pd.DataFrame(all_rows)

# Cờ chất lượng đơn giản để lọc downstream -- không xoá dòng nào, chỉ đánh dấu.
def quality_flag(row):
    if pd.isna(row.get("hr_mean_bpm")):
        return "NO_VALID_IBI"
    if row.get("segment_duration_s", 0) < 30:
        return "SHORT_SEGMENT"
    if row.get("n_valid_ibi", 0) < 10:
        return "LOW_BEAT_COUNT"
    return "OK"

ppg_hrv_long["hrv_quality_flag"] = ppg_hrv_long.apply(quality_flag, axis=1)

phase_order = ["REST_BEFORE", "MEDITATION", "REST_AFTER", "OVERALL"]
ppg_hrv_long["phase"] = pd.Categorical(ppg_hrv_long["phase"], categories=phase_order + sorted(set(ppg_hrv_long["phase"]) - set(phase_order)), ordered=True)
ppg_hrv_long = ppg_hrv_long.sort_values(["participant_key", "study_day", "phase"]).reset_index(drop=True)

# --- Gắn thêm cờ chất lượng HAPPE (EEG cùng session, tham khảo) -----------
# Chỉ số CẤP THIẾT BỊ/PHIÊN (thời lượng ghi + %% phương sai EEG giữ lại sau
# làm sạch artefact) từ eeg_happe_qc.ipynb, join theo session_id -- KHÔNG
# phải chỉ số trực tiếp đo chất lượng PPG, chỉ dùng làm biến hiệp phương
# sai/lọc độ nhạy bổ sung (cùng 1 lần đeo thiết bị). Xem lib/happe_qc.py.
import happe_qc
importlib.reload(happe_qc)
HAPPE_QC_DIR = REPO_ROOT / "data" / "analytics" / "happe_qc"
happe_quality_df = happe_qc.load_happe_quality_table(HAPPE_QC_DIR)
if happe_quality_df.empty:
    print("Chưa tìm thấy output của eeg_happe_qc.ipynb -- bỏ qua bước join (không bắt buộc).")
    for col in ["happe_file_length_s", "happe_pct_var_retained_postwavelet", "happe_pct_good_channels", "happe_qc_flags", "happe_qc_pass"]:
        ppg_hrv_long[col] = pd.NA
else:
    ppg_hrv_long = ppg_hrv_long.merge(happe_quality_df, on="session_id", how="left")
    print(f"Đã join HAPPE QC: {ppg_hrv_long['happe_qc_pass'].notna().sum()}/{len(ppg_hrv_long)} dòng khớp session_id.")

long_path = OUT_DIR / "ppg_hrv_by_phase.csv"
ppg_hrv_long.to_csv(long_path, index=False)
run_log_df.to_csv(OUT_DIR / "ppg_hrv_run_log.csv", index=False)

print(f"Đã lưu: {long_path} ({len(ppg_hrv_long)} dòng)")
print("\nPhân bố quality flag:")
print(ppg_hrv_long["hrv_quality_flag"].value_counts().to_string())
print("\nPhân bố theo phase:")
print(ppg_hrv_long["phase"].value_counts().to_string())
ppg_hrv_long.head(10)

## 6. Bảng rộng theo phase (tiện nhìn nhanh 1 session/1 dòng)

Pivot riêng -- không phải output chính, chỉ để kiểm tra nhanh. File chính để
merge với survey vẫn là bảng dài `ppg_hrv_by_phase.csv` ở trên (1 dòng =
1 session x 1 phase, đúng khoá `participant_key`+`study_day`+`session_id`+`phase`).

In [ ]:
metric_cols = ["hr_mean_bpm", "rmssd_ms", "sdrr_ms", "prr50_pct", "prr20_pct", "lf_hf_ratio"]

wide = ppg_hrv_long.pivot_table(
    index=["participant_key", "study_day", "session_id"],
    columns="phase",
    values=metric_cols,
    observed=True,
)
wide.columns = [f"{metric}__{phase}" for metric, phase in wide.columns]
wide = wide.reset_index()

wide_path = OUT_DIR / "ppg_hrv_wide_by_session.csv"
wide.to_csv(wide_path, index=False)
print(f"Đã lưu: {wide_path} ({len(wide)} dòng)")
wide.head(10)

## 7. Kiểm tra nhanh: HR/RMSSD/LF-HF theo phase

In [ ]:
ok = ppg_hrv_long[ppg_hrv_long["phase"].isin(["REST_BEFORE", "MEDITATION", "REST_AFTER"])].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col, label in zip(axes, ["hr_mean_bpm", "rmssd_ms", "lf_hf_ratio"], ["HR (bpm)", "RMSSD (ms)", "LF/HF"]):
    data = [ok.loc[ok["phase"] == p, col].dropna() for p in ["REST_BEFORE", "MEDITATION", "REST_AFTER"]]
    ax.boxplot(data, labels=["REST_BEFORE", "MEDITATION", "REST_AFTER"])
    ax.set_title(label)
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_DIR / "ppg_hrv_by_phase_boxplot.png", dpi=130)
plt.show()

print("n với LF/HF hợp lệ theo phase (cần >=60s IBI liên tục):")
print(ok.groupby("phase", observed=True)["lf_hf_ratio"].apply(lambda s: s.notna().sum()).to_string())